# Loan Approval – Machine Learning Assignment

This notebook covers:
- Logistic Regression: Normal, Ridge, and Lasso
- Decision Tree Classifier: before and after Grid Search
- Random Forest Classifier: before and after Grid Search
- A DataFrame containing exactly 7 rows
- Model evaluation using Accuracy, Precision, Recall, and F1-score

**Dataset:** `loan_approval_dataset.csv`


## 1. Import Libraries

Run each cell from top to bottom.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report


## 2. Load the Dataset

In [ ]:
df = pd.read_csv('loan_approval_dataset.csv')

# Remove extra spaces from column names and text values
df.columns = df.columns.str.strip()
for col in ['education', 'self_employed', 'loan_status']:
    df[col] = df[col].astype(str).str.strip()

print('Dataset shape:', df.shape)
display(df.head())


## 3. DataFrame with 7 Rows

The assignment asks for a DataFrame with 7 rows. We create it explicitly here. The complete dataset is still used for model training so the ML results are meaningful.

In [ ]:
df_7_rows = df.head(7).copy()
print('Number of rows:', len(df_7_rows))
display(df_7_rows)


## 4. Prepare Features and Target

`loan_status` is the target. `loan_id` is removed because it is only an identifier and does not help predict the loan decision.

In [ ]:
X = df.drop(columns=['loan_status', 'loan_id'])
y = df['loan_status'].map({'Rejected': 0, 'Approved': 1})

categorical_features = ['education', 'self_employed']
numeric_features = [col for col in X.columns if col not in categorical_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))


## 5. Preprocessing

Categorical columns are converted using One-Hot Encoding. Numeric columns are standardized for Logistic Regression.

In [ ]:
logistic_preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

tree_preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
], remainder='passthrough')


## 6. Logistic Regression – Normal, Ridge, Lasso

In [ ]:
logistic_models = {
    'Logistic Regression (Normal)': Pipeline([
        ('preprocessor', logistic_preprocessor),
        ('model', LogisticRegression(penalty=None, max_iter=2000))
    ]),
    'Logistic Regression (Ridge)': Pipeline([
        ('preprocessor', logistic_preprocessor),
        ('model', LogisticRegression(penalty='l2', C=1.0, max_iter=2000))
    ]),
    'Logistic Regression (Lasso)': Pipeline([
        ('preprocessor', logistic_preprocessor),
        ('model', LogisticRegression(penalty='l1', C=1.0, solver='liblinear', max_iter=2000))
    ])
}

results = []

for name, model in logistic_models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, predictions),
        'Precision': precision_score(y_test, predictions),
        'Recall': recall_score(y_test, predictions),
        'F1 Score': f1_score(y_test, predictions)
    })

logistic_results = pd.DataFrame(results)
display(logistic_results.round(4))


## 7. Decision Tree – Before Grid Search

In [ ]:
decision_tree = Pipeline([
    ('preprocessor', tree_preprocessor),
    ('model', DecisionTreeClassifier(random_state=42))
])

decision_tree.fit(X_train, y_train)
dt_before_pred = decision_tree.predict(X_test)

dt_before_accuracy = accuracy_score(y_test, dt_before_pred)
print('Decision Tree Accuracy BEFORE Grid Search:', round(dt_before_accuracy, 4))
print('\nClassification Report:')
print(classification_report(y_test, dt_before_pred, target_names=['Rejected', 'Approved']))


## 8. Decision Tree – After Grid Search

Grid Search tests different hyperparameter combinations and selects the best one using 5-fold cross-validation.

In [ ]:
dt_param_grid = {
    'model__criterion': ['gini', 'entropy'],
    'model__max_depth': [None, 5, 10, 15],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4]
}

dt_grid = GridSearchCV(
    decision_tree,
    param_grid=dt_param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

dt_grid.fit(X_train, y_train)
dt_after_pred = dt_grid.predict(X_test)

dt_after_accuracy = accuracy_score(y_test, dt_after_pred)
print('Best Decision Tree Parameters:')
print(dt_grid.best_params_)
print('\nBest Cross-Validation Accuracy:', round(dt_grid.best_score_, 4))
print('Decision Tree Accuracy AFTER Grid Search:', round(dt_after_accuracy, 4))


## 9. Random Forest – Before Grid Search

In [ ]:
random_forest = Pipeline([
    ('preprocessor', tree_preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

random_forest.fit(X_train, y_train)
rf_before_pred = random_forest.predict(X_test)

rf_before_accuracy = accuracy_score(y_test, rf_before_pred)
print('Random Forest Accuracy BEFORE Grid Search:', round(rf_before_accuracy, 4))
print('\nClassification Report:')
print(classification_report(y_test, rf_before_pred, target_names=['Rejected', 'Approved']))


## 10. Random Forest – After Grid Search

In [ ]:
rf_param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2]
}

rf_grid = GridSearchCV(
    random_forest,
    param_grid=rf_param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)
rf_after_pred = rf_grid.predict(X_test)

rf_after_accuracy = accuracy_score(y_test, rf_after_pred)
print('Best Random Forest Parameters:')
print(rf_grid.best_params_)
print('\nBest Cross-Validation Accuracy:', round(rf_grid.best_score_, 4))
print('Random Forest Accuracy AFTER Grid Search:', round(rf_after_accuracy, 4))


## 11. Final Comparison of All Models

In [ ]:
final_results = []

all_predictions = {
    'Logistic Regression (Normal)': logistic_models['Logistic Regression (Normal)'].predict(X_test),
    'Logistic Regression (Ridge)': logistic_models['Logistic Regression (Ridge)'].predict(X_test),
    'Logistic Regression (Lasso)': logistic_models['Logistic Regression (Lasso)'].predict(X_test),
    'Decision Tree (Before Grid Search)': dt_before_pred,
    'Decision Tree (After Grid Search)': dt_after_pred,
    'Random Forest (Before Grid Search)': rf_before_pred,
    'Random Forest (After Grid Search)': rf_after_pred
}

for name, pred in all_predictions.items():
    final_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1 Score': f1_score(y_test, pred)
    })

final_results_df = pd.DataFrame(final_results).sort_values('Accuracy', ascending=False)
display(final_results_df.round(4))


## 12. Confusion Matrix for the Best Model

The model with the highest test accuracy is selected automatically.

In [ ]:
best_model_name = final_results_df.iloc[0]['Model']
best_predictions = all_predictions[best_model_name]

print('Best Model:', best_model_name)
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, best_predictions))


## Conclusion

The final comparison shows the performance of Normal, Ridge, and Lasso Logistic Regression, plus Decision Tree and Random Forest before and after Grid Search. Grid Search is used to find better hyperparameters based on 5-fold cross-validation.

**Important:** Run all cells before submitting so the notebook contains your actual outputs, best parameters, and comparison table.